# 爬證交所資料

In [1]:
from io import StringIO
import os
import pandas as pd
import numpy as np
import json
import requests
import time
from datetime import datetime

##################################################
# 設定爬取的日期範圍和股票清單
# 以年為單位, 會覆蓋掉舊檔

start_year = 2022
end_year = 2024

data = [
    ["2330", "台積電", "tw"],
    ["2317", "鴻海", "tw"],
    ["2454", "聯發科", "tw"],
    ["2382", "廣達", "tw"],
    ["3231", "緯創", "tw"],
    ["2207", "和泰車", "tw"],
    ["2881", "富邦金", "tw"],
    ["2412", "中華電", "tw"],
    ["3045", "台灣大", "tw"],
    ["6505", "台塑化", "tw"],
    ["2603", "長榮", "tw"],
    ["6214", "精誠", "tw"],
    ["2379", "瑞昱", "tw"],
    ["3661", "世芯", "tw"],
    ["2408", "南亞科", "tw"],
    ["2357", "華碩", "tw"],
    ["2345", "智邦", "tw"],
]

##################################################

In [2]:
# 建立一個字典，其中包含所有欄位名稱的中文到英文的映射
column_name_mapping = {
    '日期': 'Date',
    '成交股數': 'Volume',
    '成交金額': 'Transaction Amount',
    '開盤價': 'Open',
    '最高價': 'High',
    '最低價': 'Low',
    '收盤價': 'Close',
    '漲跌價差': 'Diff',
    '成交筆數': 'Transaction Count'
}


def fetch_data_twse(ticker_symbol, year1, year2):
    print("Fetching data for " + ticker_symbol + " from " +
          str(year1) + " to " + str(year2) + "...")
    df_y = pd.DataFrame()
    for i in range(year1, year2+1):
        for j in range(12):
            month = str(i) + str(j+1).zfill(2)
            url = f'https://www.twse.com.tw/exchangeReport/STOCK_DAY?response=json&date={month}01&stockNo={ticker_symbol}'
            # print(url)
            data = requests.get(url).text
            data = json.loads(data)
            if data['stat'] != 'OK':
                print('Data not found for ' + month)
                break
            df_m = pd.DataFrame(data['data'])
            df_m.columns = data['fields']
            df_m = df_m.rename(columns=column_name_mapping)
            df_m.set_index('Date', inplace=True)
            df_y = pd.concat([df_y, df_m])

    # 將日期從民國年轉換為西元年
    df_y.index = pd.Series(df_y.index).str.replace(
        '/', '').apply(lambda x: str(int(x[:3]) + 1911) + x[3:])

    return df_y

# 畫技術分析


In [3]:
def cal_tech(id):

    path = os.path.join(os.path.abspath(os.path.dirname(
        os.getcwd())), 'history_data', 'tw', 'stock_price')
    # folder_list = ['2330', '2454', '2317', '2382', '2308', '3711']

    stock_tech = pd.read_csv(os.path.join(path, id + '.csv'))

    # # 將stock_tech的值都換成float
    # stock_tech['Open'] = stock_tech['Open'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Close'] = stock_tech['Close'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['High'] = stock_tech['High'].map(lambda x: float(x.replace(',', '')))
    # stock_tech['Low'] = stock_tech['Low'].map(lambda x: float(x.replace(',', '')))

    # 計算移動均線 (SMA)
    stock_tech['MA5'] = stock_tech['Close'].rolling(window=5).mean()
    stock_tech['MA10'] = stock_tech['Close'].rolling(window=10).mean()
    stock_tech['MA20'] = stock_tech['Close'].rolling(window=20).mean()

    # 計算指數移動平均線 (EMA)
    stock_tech['EMA5'] = stock_tech['Close'].ewm(span=5, adjust=False).mean()
    stock_tech['EMA12'] = stock_tech['Close'].ewm(span=12, adjust=False).mean()
    stock_tech['EMA26'] = stock_tech['Close'].ewm(span=26, adjust=False).mean()

    # 計算MACD和DIF
    stock_tech['DIF'] = stock_tech['EMA12'] - stock_tech['EMA26']
    stock_tech['MACD'] = stock_tech['DIF'].ewm(span=9, adjust=False).mean()

    # 計算RSI

    def calculate_rsi(series, period=14):
        delta = series.diff(1)
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi

    stock_tech['RSI'] = calculate_rsi(stock_tech['Close'])

    # 計算KDJ
    low_list = stock_tech['Low'].rolling(9, min_periods=9).min()
    low_list.fillna(value=stock_tech['Low'].expanding().min(), inplace=True)
    high_list = stock_tech['High'].rolling(9, min_periods=9).max()
    high_list.fillna(value=stock_tech['High'].expanding().max(), inplace=True)
    rsv = (stock_tech['Close'] - low_list) / (high_list - low_list) * 100

    # 初始值設定
    stock_tech['K'] = 50
    stock_tech['D'] = 50

    for i in range(1, len(stock_tech)):
        stock_tech.loc[stock_tech.index[i], 'K'] = (
            2/3) * stock_tech.loc[stock_tech.index[i-1], 'K'] + (1/3) * rsv[i]
        stock_tech.loc[stock_tech.index[i], 'D'] = (
            2/3) * stock_tech.loc[stock_tech.index[i-1], 'D'] + (1/3) * stock_tech.loc[stock_tech.index[i], 'K']

    # 顯示DataFrame
    stock_tech.to_csv(os.path.join(path, id + 'tech.csv'),
                      encoding='utf-8', index=False)

# 主迴圈

In [4]:
path = os.path.join(os.path.abspath(os.path.dirname(os.getcwd())), 'history_data', 'tw', 'stock_price')
for stock_data in data:
    id = stock_data[0]
    
    try:
        stock_tech = fetch_data_twse(id, 2022, 2024)  # 這裡可以自行設定要抓取的時間範圍
        # Remove commas and convert to numeric types
        df = stock_tech.map(lambda x: pd.to_numeric(
            str(x).replace(",", "").replace('"', ''), errors='coerce'))
        df.to_csv(os.path.join(path, id + '.csv'))  # 將資料存成 csv 檔
        print('save ' + id + '.csv')
        cal_tech(id)
        print('save ' + id + 'tech.csv')
    except Exception as e:
        print("Error: " + id + " " + str(e))
        pass

Fetching data for 2330 from 2022 to 2024...
save 2330.csv
save 2330tech.csv
Fetching data for 2317 from 2022 to 2024...


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '66.66666666666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '55.55555555555554' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (


save 2317.csv
save 2317tech.csv
Fetching data for 2454 from 2022 to 2024...


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '41.66666666666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '47.222222222222214' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (


save 2454.csv
save 2454tech.csv
Fetching data for 2382 from 2022 to 2024...


...

/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.33333333333337' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '51.111111111111114' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (


save 2357.csv
save 2357tech.csv
Fetching data for 2345 from 2022 to 2024...


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '59.52380952380952' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.17460317460316' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (


save 2345.csv
save 2345tech.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:53: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '60.41666666666666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'K'] = (
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_77493/3368124125.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.472222222222214' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  stock_tech.loc[stock_tech.index[i], 'D'] = (
